

### 3.2 Vector Search + RAG Fusion

### 3.3 Hybrid search + Reranking



In [1]:
import os
import numpy as np
import pandas as pd

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from financerag.task import *
from financerag.retrieval import DenseRetrieval
from langchain_text_splitters.base import TextSplitter
pd.set_option('display.max_colwidth', 400)
%reload_ext autoreload

In [ ]:
load_dotenv(".env")
GG_API_KEY = os.environ.get('GOOGLE_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')

In [4]:
# Set up vector database
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004")
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [4]:
import torch # type: ignore
torch.__version__

'2.3.0.dev20240311'

In [7]:
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


In [3]:
# Get dataset name first
import os
dataset_names = []
for f in os.listdir('finance_dataset'):
    if f.endswith("tsv"):
       dataset_names.append(f.split('_')[0])
dataset_names  

['MultiHeirtt',
 'FinQA',
 'FinanceBench',
 'ConvFinQA',
 'FinQABench',
 'TATQA',
 'FinDER']

In [10]:
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 300
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

In [ ]:
# Idea for optimizing the retrieved result
# Find out the largest possible length in each corpus
# Take # of splits = maxLen / (chunk_size - chunk_overlap)
# Top_k must be (#splits + 1) * 10 -> max(doc_size +)
# Test to see result first!

In [15]:
df = pd.read_json('finance_dataset/tatqa_corpus.jsonl/corpus.jsonl', lines = True)
df['text'].str.len().max()

14625

In [31]:
import logging 

logging.info('hellow rod')

In [23]:
dataset_name = dataset_names[0]
task_variable = f"{dataset_name.lower()}_task"
script_string = f"""
# {dataset_name} Task
print(f"{dataset_name} task")
{task_variable} = {dataset_name}Task()
{task_variable}.load()
{task_variable}_max_len = np.max([len(text) for text in {task_variable}.corpus.values()])
{task_variable}_top_k = ({task_variable}_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10
{task_variable}_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = {task_variable}.metadata.dataset_name)
{task_variable}_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = {task_variable}.corpus, saved_index = True)
{task_variable}.retrieve(retriever = {task_variable}_retriever, top_k = {task_variable}_top_k)
{task_variable}.save_retrieved_results()
"""
print(script_string)


# MultiHeirtt Task
print(f"MultiHeirtt task")
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load()
multiheirtt_task_max_len = np.max([len(text) for text in multiheirtt_task.corpus.values()])
multiheirtt_task_top_k = (multiheirtt_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10
print("Top k for multiheirtt_task:" multiheirtt_task_top_k)



In [37]:
# MultiHeirtt Task
print(f"MultiHeirtt task")
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load()
multiheirtt_task_max_len = np.max([len(text) for text in multiheirtt_task.corpus.values()])
multiheirtt_task_top_k = (multiheirtt_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
print("Top k for multiheirtt_task:", multiheirtt_task_top_k)

MultiHeirtt task
Top k for multiheirtt_task: 120


In [38]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    print(f"{dataset_name} task")
    {task_variable} = {dataset_name}Task()
    {task_variable}.load()
    {task_variable}_max_len = np.max([len(text) for text in {task_variable}.corpus.values()])
    {task_variable}_top_k = ({task_variable}_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
    {task_variable}_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = {task_variable}.metadata.dataset_name)
    {task_variable}_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = {task_variable}.corpus, saved_index = False)
    {task_variable}.retrieve(retriever = {task_variable}_retriever, top_k = {task_variable}_top_k)
    {task_variable}.save_retrieved_results()
    """
    print(script_string)


    # MultiHeirtt Task
    print(f"MultiHeirtt task")
    multiheirtt_task = MultiHeirttTask()
    multiheirtt_task.load()
    multiheirtt_task_max_len = np.max([len(text) for text in multiheirtt_task.corpus.values()])
    multiheirtt_task_top_k = (multiheirtt_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
    multiheirtt_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = multiheirtt_task.metadata.dataset_name)
    multiheirtt_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = multiheirtt_task.corpus, saved_index = False)
    multiheirtt_task.retrieve(retriever = multiheirtt_task_retriever, top_k = multiheirtt_task_top_k)
    multiheirtt_task.save_retrieved_results()
    

    # FinQA Task
    print(f"FinQA task")
    finqa_task = FinQATask()
    finqa_task.load()
    finqa_task_max_len = np.max([len(text) for text in finqa_task.corpus.values()])
    finqa_task_top_k = (finqa_task_max_len // (CHUNK_SIZE - CHUNK_OVERL

In [39]:
# MultiHeirtt Task
print(f"MultiHeirtt task")
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load()
multiheirtt_task_max_len = np.max([len(text) for text in multiheirtt_task.corpus.values()])
multiheirtt_task_top_k = (multiheirtt_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
multiheirtt_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = multiheirtt_task.metadata.dataset_name)
multiheirtt_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = multiheirtt_task.corpus, saved_index = False)
multiheirtt_task.retrieve(retriever = multiheirtt_task_retriever, top_k = multiheirtt_task_top_k)
multiheirtt_task.save_retrieved_results()


# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load()
finqa_task_max_len = np.max([len(text) for text in finqa_task.corpus.values()])
finqa_task_top_k = (finqa_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
finqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finqa_task.corpus, saved_index = False)
finqa_task.retrieve(retriever = finqa_task_retriever, top_k = finqa_task_top_k)
finqa_task.save_retrieved_results()


# FinanceBench Task
print(f"FinanceBench task")
financebench_task = FinanceBenchTask()
financebench_task.load()
financebench_task_max_len = np.max([len(text) for text in financebench_task.corpus.values()])
financebench_task_top_k = (financebench_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
financebench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = financebench_task.metadata.dataset_name)
financebench_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = financebench_task.corpus, saved_index = False)
financebench_task.retrieve(retriever = financebench_task_retriever, top_k = financebench_task_top_k)
financebench_task.save_retrieved_results()


# ConvFinQA Task
print(f"ConvFinQA task")
convfinqa_task = ConvFinQATask()
convfinqa_task.load()
convfinqa_task_max_len = np.max([len(text) for text in convfinqa_task.corpus.values()])
convfinqa_task_top_k = (convfinqa_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
convfinqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = convfinqa_task.metadata.dataset_name)
convfinqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = convfinqa_task.corpus, saved_index = False)
convfinqa_task.retrieve(retriever = convfinqa_task_retriever, top_k = convfinqa_task_top_k)
convfinqa_task.save_retrieved_results()


# FinQABench Task
print(f"FinQABench task")
finqabench_task = FinQABenchTask()
finqabench_task.load()
finqabench_task_max_len = np.max([len(text) for text in finqabench_task.corpus.values()])
finqabench_task_top_k = (finqabench_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
finqabench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqabench_task.metadata.dataset_name)
finqabench_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finqabench_task.corpus, saved_index = False)
finqabench_task.retrieve(retriever = finqabench_task_retriever, top_k = finqabench_task_top_k)
finqabench_task.save_retrieved_results()


# TATQA Task
print(f"TATQA task")
tatqa_task = TATQATask()
tatqa_task.load()
tatqa_task_max_len = np.max([len(text) for text in tatqa_task.corpus.values()])
tatqa_task_top_k = (tatqa_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
tatqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = tatqa_task.metadata.dataset_name)
tatqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = tatqa_task.corpus, saved_index = False)
tatqa_task.retrieve(retriever = tatqa_task_retriever, top_k = tatqa_task_top_k)
tatqa_task.save_retrieved_results()


# FinDER Task
print(f"FinDER task")
finder_task = FinDERTask()
finder_task.load()
finder_task_max_len = np.max([len(text) for text in finder_task.corpus.values()])
finder_task_top_k = (finder_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
finder_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finder_task.metadata.dataset_name)
finder_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finder_task.corpus, saved_index = False)
finder_task.retrieve(retriever = finder_task_retriever, top_k = finder_task_top_k)
finder_task.save_retrieved_results()

MultiHeirtt task


Retrieving result::  30%|██▉       | 290/974 [39:06<1:32:14,  8.09s/it]


GoogleGenerativeAIError: Error embedding content: 504 Deadline Exceeded

In [1]:
from sentence_transformers import CrossEncoder

/Users/mac/Desktop/Code/Personal_Project/DeepLearningProject/RAG_Project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from nltk.tokenize import word_tokenize
from langchain_community.retrievers import BM25Retriever
retriever = BM25Retriever.from_documents(
    [
        Document(page_content="foo"),
        Document(page_content="bar"),
        Document(page_content="world"),
        Document(page_content="hello"),
        Document(page_content="foo bar"),
    ],
    k=2,
    preprocess_func=word_tokenize,
)

result = retriever.invoke("bar")
result

[Document(metadata={}, page_content='bar'),
 Document(metadata={}, page_content='foo bar')]

In [2]:
# 1. Load a pretrained CrossEncoder model
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", device = 'cpu')

# The texts for which to predict similarity scores
query = "How many people live in Berlin?"
passages = [
    "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers.",
    "Berlin has a yearly total of about 135 million day visitors, making it one of the most-visited cities in the European Union.",
    "In 2013 around 600,000 Berliners were registered in one of the more than 2,300 sport and fitness clubs.",
]

# 2a. Either predict scores pairs of texts
scores = model.predict([(query, passage) for passage in passages])
print(scores)
# => [8.607139 5.506266 6.352977]

# 2b. Or rank a list of passages for a query
ranks = model.rank(query, passages, return_documents=True)

print("Query:", query)
for rank in ranks:
    print(f"- #{rank['corpus_id']} ({rank['score']:.2f}): {rank['text']}")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
[8.607143  5.506265  6.3529806]
Query: How many people live in Berlin?
- #0 (8.61): Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers.
- #2 (6.35): In 2013 around 600,000 Berliners were registered in one of the more than 2,300 sport and fitness clubs.
- #1 (5.51): Berlin has a yearly total of about 135 million day visitors, making it one of the most-visited cities in the European Union.


In [ ]:
# MultiHeirtt Task
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load()
multiheirtt_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = multiheirtt_task.metadata.dataset_name)
multiheirtt_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = multiheirtt_task.corpus, saved_index = True)
multiheirtt_task.retrieve(retriever = multiheirtt_task_retriever)
multiheirtt_task.save_retrieved_results()



Loading document:: 100%|██████████| 10475/10475 [00:00<00:00, 42579.02it/s]


Successfully saved index of vector store to path : faiss_index/multiheirtt_index
Saved result successfully to ./financerag_result/multiheirtt_result.csv!


In [ ]:
# FinQA Task
finqa_task = FinQATask()
finqa_task.load()
finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
finqa_task_retriever.load_corpus_for_searching_without_splitting(finqa_task.corpus, saved_index = True)
finqa_task.retrieve(retriever = finqa_task_retriever)
finqa_task.save_retrieved_results()




Loading document: 100%|██████████| 2789/2789 [00:00<00:00, 151775.10it/s]


Successfully saved index of vector store to path : faiss_index/finqa_index


Retrieving result:: 100%|██████████| 1147/1147 [07:21<00:00,  2.60it/s]

Saved result successfully to ./financerag_result/finqa_result.csv!


In [13]:

# FinanceBench Task
financebench_task = FinanceBenchTask()
financebench_task.load()
financebench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = financebench_task.metadata.dataset_name)
financebench_task_retriever.load_corpus_for_searching(financebench_task.corpus, saved_index = True)
financebench_task.retrieve(retriever = financebench_task_retriever)
financebench_task.save_retrieved_results()




Loading document: 100%|██████████| 180/180 [00:00<00:00, 131942.45it/s]


Successfully saved index of vector store to path : faiss_index/financebench_index


Retrieving result:: 100%|██████████| 150/150 [01:02<00:00,  2.42it/s]

Saved result successfully to ./financerag_result/financebench_result.csv!


In [ ]:
# ConvFinQA Task
convfinqa_task = ConvFinQATask()
convfinqa_task.load()
convfinqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = convfinqa_task.metadata.dataset_name)
convfinqa_task_retriever.load_corpus_for_searching_without_splitting(convfinqa_task.corpus, saved_index = True)
convfinqa_task.retrieve(retriever = convfinqa_task_retriever)
convfinqa_task.save_retrieved_results()


Loading document: 100%|██████████| 2066/2066 [00:00<00:00, 129481.68it/s]


Successfully saved index of vector store to path : faiss_index/convfinqa_index


Retrieving result:: 100%|██████████| 421/421 [02:37<00:00,  2.67it/s]

Saved result successfully to ./financerag_result/convfinqa_result.csv!


In [15]:

# FinQABench Task
finqabench_task = FinQABenchTask()
finqabench_task.load()
finqabench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqabench_task.metadata.dataset_name)
finqabench_task_retriever.load_corpus_for_searching(finqabench_task.corpus, saved_index = True)
finqabench_task.retrieve(retriever = finqabench_task_retriever)
finqabench_task.save_retrieved_results()


Loading document: 100%|██████████| 92/92 [00:00<00:00, 89592.75it/s]


Successfully saved index of vector store to path : faiss_index/finqabench_index


Retrieving result:: 100%|██████████| 100/100 [00:36<00:00,  2.75it/s]

Saved result successfully to ./financerag_result/finqabench_result.csv!


In [ ]:

# TATQA Task
tatqa_task = TATQATask()
tatqa_task.load()
tatqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = tatqa_task.metadata.dataset_name)
tatqa_task_retriever.load_corpus_for_searching_without_splitting(tatqa_task.corpus, saved_index = True)
tatqa_task.retrieve(retriever = tatqa_task_retriever)
tatqa_task.save_retrieved_results() 


Loading document: 100%|██████████| 2756/2756 [00:00<00:00, 182551.12it/s]


Successfully saved index of vector store to path : faiss_index/tatqa_index


Retrieving result:: 100%|██████████| 1663/1663 [15:14<00:00,  1.82it/s] 

Saved result successfully to ./financerag_result/tatqa_result.csv!


In [ ]:

# FinDER Task
finder_task = FinDERTask()
finder_task.load()
finder_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finder_task.metadata.dataset_name)
finder_task_retriever.load_corpus_for_searching_without_splitting(finder_task.corpus, saved_index = True)
finder_task.retrieve(retriever = finder_task_retriever)
finder_task.save_retrieved_results()

Loading document: 100%|██████████| 13862/13862 [00:00<00:00, 20056.98it/s]


Successfully saved index of vector store to path : faiss_index/finder_index


Retrieving result:: 100%|██████████| 216/216 [01:57<00:00,  1.84it/s]

Saved result successfully to ./financerag_result/finder_result.csv!


In [16]:
final_result = pd.DataFrame(columns = ['query_id', 'corpus_id'])

for dataset_name in dataset_names:
    df = pd.read_csv(f"financerag_result/{dataset_name.lower()}_result.csv")
    final_result = pd.concat([final_result, df], axis = 0)

final_result

,query_id,corpus_id
0,q82d4c6ec,d8177a896
1,q82d4c6ec,d8d3fbbaa
2,q82d4c6ec,d85eb42c0
3,q82d4c6ec,d8e817e40
4,q82d4c6ec,d8bb12b52
...,...,...
2155,q00218,ADBE20230512
2156,q00218,TSLA20231025
2157,q00218,CPNG20230812
2158,q00218,LIN20230185


In [17]:
pd.read_csv('submission.csv')

,query_id,corpus_id
0,q82d4c6ec,d8e404704
1,q82d4c6ec,d87914156
2,q82d4c6ec,d8a3219a8
3,q82d4c6ec,d8c22a07a
4,q82d4c6ec,d88e1c5b2
...,...,...
44538,q00218,GOOGL20231203
44539,q00218,JPM20234729
44540,q00218,V20231584
44541,q00218,JPM20237319


In [20]:
pd.read_csv('finance_dataset/sample_submission_.csv')

,query_id,corpus_id
0,qd496c6a0,dd4b92b32
1,qd496c6a0,dd4ba2a5a
2,qd496c6a0,dd4be1f98
3,qd496c6a0,dd4ba07d2
4,qd496c6a0,dd4ba02f0
...,...,...
46681,q1a741e68,d1b3afaba
46682,q1a741e68,d1b34e18e
46683,q1a741e68,d1b36065e
46684,q1a741e68,d1b33d05a


In [18]:
final_result.to_csv('submission_bm25.csv', index = False)

In [22]:
final_result.drop_duplicates(subset = ['query_id'])

,query_id,corpus_id
0,q82d4c6ec,d8e404704
10,q855a35a0,d89a6ea36
20,q85384530,d8ce7fc30
30,q842c8af2,d88465f0a
40,q85451756,d8646ec7e
...,...,...
2094,q00214,BRK.A20230009
2104,q00215,BRK.A20230404
2114,q00216,BRK.A20232401
2124,q00217,BRK.A20230062


In [28]:
final_queries = pd.DataFrame(columns = ['_id', 'title', 'text'])

for dataset_name in dataset_names:
    df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_queries.jsonl/queries.jsonl", lines = True)
    final_queries = pd.concat([final_queries, df], axis = 0)

final_queries

,_id,title,text
0,q82d4c6ec,,What was the sum of Fourth Quarter without tho...
1,q855a35a0,,In which section is Interest income smaller th...
2,q85384530,,If Total Forward Hedged Revenues develops with...
3,q842c8af2,,what was the ratio of the purchase in december...
4,q85451756,,what is the highest total amount of segment in...
...,...,...,...
211,q00214,,How many distinct insurance underwriting group...
212,q00215,,What is the ticker symbol for Berkshire Hathaw...
213,q00216,,What is the largest operating segment of the B...
214,q00217,,Source of invested assets of insurance busines...


In [7]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    print(f"{dataset_name} task")
    {task_variable} = {dataset_name}Task()
    {task_variable}.load()
    {task_variable}_bm25_retriever = BM25_Retriever()
    retrieved_result = {task_variable}.retrieve(retriever = {task_variable}_bm25_retriever, corpus = {task_variable}.corpus, top_k = 10)
    {task_variable}.save_retrieved_results(retrieved_result = retrieved_result)
    """
    print(script_string)


    # MultiHeirtt Task
    print(f"MultiHeirtt task")
    multiheirtt_task = MultiHeirttTask()
    multiheirtt_task.load()
    multiheirtt_task_bm25_retriever = BM25_Retriever()
    retrieved_result = multiheirtt_task.retrieve(retriever = multiheirtt_task_bm25_retriever, corpus = multiheirtt_task.corpus, top_k = 10)
    multiheirtt_task.save_retrieved_results(retrieved_result = retrieved_result)
    

    # FinQA Task
    print(f"FinQA task")
    finqa_task = FinQATask()
    finqa_task.load()
    finqa_task_bm25_retriever = BM25_Retriever()
    retrieved_result = finqa_task.retrieve(retriever = finqa_task_bm25_retriever, corpus = finqa_task.corpus, top_k = 10)
    finqa_task.save_retrieved_results(retrieved_result = retrieved_result)
    

    # FinanceBench Task
    print(f"FinanceBench task")
    financebench_task = FinanceBenchTask()
    financebench_task.load()
    financebench_task_bm25_retriever = BM25_Retriever()
    retrieved_result = financebench_task.retrieve(retriever = fin

In [5]:
from financerag.retrieval import BM25, BM25_Retriever

In [7]:

# MultiHeirtt Task
# print(f"MultiHeirtt task")
# multiheirtt_task = MultiHeirttTask()
# multiheirtt_task.load()
# multiheirtt_task_bm25_retriever = BM25_Retriever()
# multiheirtt_task.retrieve(retriever = multiheirtt_task_bm25_retriever, corpus = multiheirtt_task.corpus, top_k = 10)
# multiheirtt_task.save_retrieved_results()


# # FinQA Task
# print(f"FinQA task")
# finqa_task = FinQATask()
# finqa_task.load()
# finqa_task_bm25_retriever = BM25_Retriever()
# finqa_task.retrieve(retriever = finqa_task_bm25_retriever, corpus = finqa_task.corpus, top_k = 10)
# finqa_task.save_retrieved_results()



# FinanceBench Task
print(f"FinanceBench task")
financebench_task = FinanceBenchTask()
financebench_task.load()
financebench_task_bm25_retriever = BM25_Retriever()
retrieved_result = financebench_task.retrieve(retriever = financebench_task_bm25_retriever, corpus = financebench_task.corpus, top_k = 10)
financebench_task.save_retrieved_results(retrieved_result = retrieved_result)


# ConvFinQA Task
print(f"ConvFinQA task")
convfinqa_task = ConvFinQATask()
convfinqa_task.load()
convfinqa_task_bm25_retriever = BM25_Retriever()
retrieved_result = convfinqa_task.retrieve(retriever = convfinqa_task_bm25_retriever, corpus = convfinqa_task.corpus, top_k = 10)
convfinqa_task.save_retrieved_results(retrieved_result = retrieved_result)


# FinQABench Task
print(f"FinQABench task")
finqabench_task = FinQABenchTask()
finqabench_task.load()
finqabench_task_bm25_retriever = BM25_Retriever()
retrieved_result = finqabench_task.retrieve(retriever = finqabench_task_bm25_retriever, corpus = finqabench_task.corpus, top_k = 10)
finqabench_task.save_retrieved_results(retrieved_result = retrieved_result)


# TATQA Task
print(f"TATQA task")
tatqa_task = TATQATask()
tatqa_task.load()
tatqa_task_bm25_retriever = BM25_Retriever()
retrieved_result = tatqa_task.retrieve(retriever = tatqa_task_bm25_retriever, corpus = tatqa_task.corpus, top_k = 10)
tatqa_task.save_retrieved_results(retrieved_result = retrieved_result)


# FinDER Task
print(f"FinDER task")
finder_task = FinDERTask()
finder_task.load()
finder_task_bm25_retriever = BM25_Retriever()
retrieved_result = finder_task.retrieve(retriever = finder_task_bm25_retriever, corpus = finder_task.corpus, top_k = 10)
finder_task.save_retrieved_results(retrieved_result = retrieved_result)

FinanceBench task
Saved result successfully to ./financerag_result/financebench_result.csv!
ConvFinQA task
Saved result successfully to ./financerag_result/convfinqa_result.csv!
FinQABench task
Saved result successfully to ./financerag_result/finqabench_result.csv!
TATQA task
Saved result successfully to ./financerag_result/tatqa_result.csv!
FinDER task
Saved result successfully to ./financerag_result/finder_result.csv!


In [10]:
retrieved_result

{'qd2ac917a': {'dd2adb8d4': 'Exhibit Johnson Johnson Announces Updated Financials and 2023 Guidance Following Completion of the Kenvue Separation Company expects increased 2023 Reported Sales Growth of Operational Sales Growth of and Adjusted Operational Sales Growth of Figures exclude the COVID Vaccine Company expects 2023 Adjusted Reported Earnings Per Share EPS of reflecting increased growth of at the mid point and Adjusted Operational EPS of reflecting increased growth of at the mid point Company reduced outstanding share count by approximately million 2023 guidance reflects only a partial year benefit of approximately million shares or benefit to EPS Company secured billion in cash proceeds from the Kenvue debt offering and initial public offering and maintains of equity stake in Kenvue Company maintains its quarterly dividend of per share New Brunswick N J August 2023 Johnson Johnson NYSE JNJ the Company today announced updates to its financials and 2023 guidance which reflect it

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /Users/mac/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [12]:
sent = multiheirtt_task.queries['q81e448f2'].lower()

stop_words = set(stopwords.words('english'))

word_tokens = word_tokenize(sent)
word_tokens

['considering',
 'the',
 'years',
 '2015-2016',
 ',',
 'what',
 'was',
 'the',
 'decrease',
 'observed',
 'in',
 'the',
 'expense',
 'for',
 'severance',
 'and',
 'other',
 'benefits',
 '?']

TypeError: list indices must be integers or slices, not tuple

In [ ]:
im

ModuleNotFoundError: No module named 'nltk'